# Notebook to convert videos between .mov and .mp4, optionally shrinking the file size

In [1]:
import os, sys, shutil, subprocess, tempfile, glob, json

## Configuration

In [2]:
# Choose the input video and desired conversion direction
conversion_direction = 'mp4_to_mov'  # Use 'mov_to_mp4' to convert the other way around
input_file = '/Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/Publishing/Publications/Metabolism/WileySubmission/MultimediaMaterial/hive1_AbruptDay.mp4'

# How much smaller you'd like the output file to be compared to the input, as a percentage.
#   0   -> no re-encoding, just remux/convert the container (identical quality)
#   50  -> aim for roughly half the input file size (quality is lowered to get there)
#   90  -> aim for a strong reduction (quality will be noticeably lower)
# This is a target, not a guarantee: actual output size depends on the video content.
target_size_reduction_percent = 65

input_ext = os.path.splitext(input_file)[1].lower()
if conversion_direction == 'mov_to_mp4':
    expected_input_ext = '.mov'
    output_ext = '.mp4'
elif conversion_direction == 'mp4_to_mov':
    expected_input_ext = '.mp4'
    output_ext = '.mov'
else:
    raise ValueError("conversion_direction must be either 'mov_to_mp4' or 'mp4_to_mov'")

if input_ext != expected_input_ext:
    raise ValueError(f'For {conversion_direction}, the input file must end with {expected_input_ext}, got {input_ext}')

if not (0 <= target_size_reduction_percent <= 95):
    raise ValueError('target_size_reduction_percent must be between 0 and 95')

output_file = os.path.splitext(input_file)[0] + output_ext

# If the output file already exists, append an increasing counter: "{name}_1", "{name}_2", ...
if os.path.exists(output_file):
    base_name = os.path.splitext(input_file)[0]
    counter = 1
    while os.path.exists(f'{base_name}_{counter}{output_ext}'):
        counter += 1
    output_file = f'{base_name}_{counter}{output_ext}'

## Main code

In [3]:
def _find_binary(name):
    """Locate an ffmpeg/ffprobe executable: first on PATH, then next to the current
    Python interpreter (covers conda-env kernels where PATH isn't fully activated)."""
    path = shutil.which(name)
    if path:
        return path
    candidate = os.path.join(os.path.dirname(sys.executable), name)
    if os.path.exists(candidate):
        return candidate
    raise FileNotFoundError(
        f"Could not find '{name}'. Install it in this environment, e.g. "
        f"`conda install -c conda-forge ffmpeg`, and make sure it's on PATH."
    )


def _run(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n\n{result.stderr}")
    return result


ffmpeg_bin = _find_binary('ffmpeg')
ffprobe_bin = _find_binary('ffprobe')

if not os.path.exists(input_file):
    raise FileNotFoundError(f'Could not open input video file: {input_file}')

# Probe the input file for duration/bitrate and whether it has an audio stream
probe = _run([
    ffprobe_bin, '-v', 'error',
    '-show_entries', 'format=duration,bit_rate:stream=codec_type',
    '-of', 'json', input_file,
])
info = json.loads(probe.stdout)
duration = float(info.get('format', {}).get('duration') or 0)
orig_bitrate = info.get('format', {}).get('bit_rate')
has_audio = any(s.get('codec_type') == 'audio' for s in info.get('streams', []))

input_size_bytes = os.path.getsize(input_file)
if orig_bitrate:
    orig_bitrate = int(orig_bitrate)
elif duration > 0:
    orig_bitrate = int(input_size_bytes * 8 / duration)
else:
    raise RuntimeError('Could not determine input bitrate or duration from the input file')

if target_size_reduction_percent == 0:
    # No quality loss requested: just remux into the new container
    _run([ffmpeg_bin, '-y', '-i', input_file, '-c', 'copy', output_file])
else:
    target_total_bitrate = int(orig_bitrate * (1 - target_size_reduction_percent / 100))

    audio_bitrate = 128_000 if has_audio else 0
    min_video_bitrate = 200_000  # floor so we never ask for a degenerate, unwatchable bitrate
    video_bitrate = max(target_total_bitrate - audio_bitrate, min_video_bitrate)

    passlog_prefix = os.path.join(tempfile.gettempdir(), f'ffmpeg2pass_{os.getpid()}')
    video_args = ['-c:v', 'libx264', '-b:v', str(video_bitrate), '-pix_fmt', 'yuv420p']

    try:
        # Pass 1: analyze, write no output
        _run([
            ffmpeg_bin, '-y', '-i', input_file,
            *video_args, '-pass', '1', '-passlogfile', passlog_prefix,
            '-an', '-f', 'null', os.devnull,
        ])

        # Pass 2: actually encode, using the stats gathered above
        pass2_cmd = [
            ffmpeg_bin, '-y', '-i', input_file,
            *video_args, '-pass', '2', '-passlogfile', passlog_prefix,
        ]
        if has_audio:
            pass2_cmd += ['-c:a', 'aac', '-b:a', str(audio_bitrate)]
        else:
            pass2_cmd += ['-an']
        if output_ext == '.mp4':
            pass2_cmd += ['-movflags', '+faststart']
        pass2_cmd += [output_file]
        _run(pass2_cmd)
    finally:
        for f in glob.glob(passlog_prefix + '*'):
            os.remove(f)

output_size_bytes = os.path.getsize(output_file)
actual_reduction_percent = 100 * (1 - output_size_bytes / input_size_bytes)

print(f'Video converted and saved as: {output_file}')
print(f'Input size:  {input_size_bytes / 1e6:.2f} MB')
print(f'Output size: {output_size_bytes / 1e6:.2f} MB')
print(f'Size change: {actual_reduction_percent:.1f}% smaller (target was {target_size_reduction_percent}%)')

Video converted and saved as: /Users/cyrilmonette/Desktop/EPFL 2018-2026/PhD - Mobots/Publishing/Publications/Metabolism/WileySubmission/MultimediaMaterial/hive1_AbruptDay.mov
Input size:  1303.49 MB
Output size: 453.67 MB
Size change: 65.2% smaller (target was 65%)
